# OXIOW — EchoMimicV3-Flash na T4 grátis (Colab)

**Objetivo:** avatar falante (lipsync) gerado de graça, para decidir se precisamos
alugar GPU (Vast.ai) ou não.

**Antes de rodar:** menu **Runtime → Change runtime type → T4 GPU → Save**.
(O Colab grátis entrega T4. Se vier K80/P100, esta célula 1 vai avisar.)

**Por que aqui e não no Kaggle:** a CLI do Kaggle **não** consegue escolher a GPU —
entrega P100 (sm_60), que o PyTorch do Kaggle **não suporta**. Testado 2× com
`accelerator` e com `machine_shape`: as duas foram ignoradas.

**Critério:** gerar 1 clipe falante em ≤ 20 min.


In [ ]:
# ── 1. AMBIENTE ──
import os, sys, subprocess, shutil, time, glob

def sh(cmd, timeout=5400, quiet=False):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    out = (p.stdout or '') + (p.stderr or '')
    if not quiet:
        print('\n'.join(out.strip().splitlines()[-12:]))
    return p.returncode, out

print('=== GPU ===')
sh('nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader')

import torch
nome = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''
cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
print('\ntorch:', torch.__version__)
print('cuDNN:', torch.backends.cudnn.version())
print('GPU:', nome, '| capability:', cap, '(sm_75 = T4, servico)')

falta_disco = shutil.disk_usage('/content').free / 1e9
print(f'\ndisco /content livre: {falta_disco:.1f} GB')

if not torch.cuda.is_available():
    print('\n>>> SEM GPU. Runtime > Change runtime type > T4 GPU <<<')
elif cap[0] < 7:
    print(f'\n>>> GPU FRACA (sm_{cap[0]}): o PyTorch do Colab nao suporta Pascal.')
    print('>>> Nao resolva com "Run anyway": nem comecar. Se persistir, use Vast.ai.')
else:
    os.makedirs('/content/em3', exist_ok=True)
    print('\n=== CHECKPOINT 1: OK ===')


In [ ]:
# ── 2. REPO + DEPENDENCIAS (lista COMPLETA do requirements oficial) ──
# Licao do Kaggle: instalar parcialmente quebra em 'import decord' e no moviepy 1.x
# (o repo usa a API da 2.x). Aqui vai a lista inteira, com decord em 2 tentativas.
sh('cd /content/em3 && [ -d echomimic_v3 ] || git clone --depth 1 '
   'https://github.com/antgroup/echomimic_v3.git', timeout=1200)

sh('pip install -q "diffusers>=0.30.1" "transformers>=4.46.2" "accelerate>=0.25.0" '
   'omegaconf safetensors einops timm tomesd imageio imageio-ffmpeg opencv-python-headless '
   'scikit-image sentencepiece ftfy beautifulsoup4 Pillow datasets torchdiffeq torchsde '
   'albumentations func_timeout onnxruntime librosa pyloudnorm soundfile '
   '"moviepy==2.2.1" 2>&1 | tail -6', timeout=3000)

sh('''
python -c "import decord" 2>/dev/null && echo "decord OK" && exit 0
echo ">>> instalando decord <<<"
pip install -q decord 2>&1 | tail -2
python -c "import decord" 2>/dev/null && echo "decord OK" && exit 0
echo ">>> decord falhou, tentando eva-decord <<<"
pip install -q eva-decord 2>&1 | tail -2
python -c "import decord; print('decord', decord.__version__)" 2>&1 | tail -1
''', timeout=1800)

print('\n=== CONFERENCIA DOS IMPORTS CRITICOS ===')
faltando = []
for m in ['decord', 'librosa', 'pyloudnorm', 'moviepy', 'torchdiffeq', 'torchsde',
          'einops', 'omegaconf', 'diffusers', 'transformers', 'accelerate', 'tomesd']:
    rc, _ = sh(f'python -c "import {m}"', timeout=180, quiet=True)
    print(f'   {"OK   " if rc == 0 else "FALTA"} {m}')
    if rc != 0:
        faltando.append(m)
print('\n=== CHECKPOINT 2:', 'OK' if not faltando else f'FALTAM: {faltando}', '===')

# o moviepy 2.x mudou a API; o repo importa 'from moviepy import VideoFileClip'
rc, out = sh('python -c "from moviepy import VideoFileClip, AudioFileClip; print(moviepy_ok)" 2>&1 | tail -2')


In [ ]:
# ── 3. PESOS (~10-13 GB) ──
# API Python, nao CLI: 'huggingface-cli' virou 'hf' e quebraria calado.
from huggingface_hub import snapshot_download
import traceback

BASE = '/content/em3/pesos'
os.makedirs(BASE, exist_ok=True)
t0 = time.time()

tarefas = [
    ('a) Wan2.1-Fun-V1.1-1.3B-InP (backbone)',
     'alibaba-pai/Wan2.1-Fun-V1.1-1.3B-InP', f'{BASE}/Wan2.1-Fun-V1.1-1.3B-InP',
     ['*.md', '*.txt']),
    ('b) chinese-wav2vec2-base (encoder de audio)',
     'TencentGameMate/chinese-wav2vec2-base', f'{BASE}/chinese-wav2vec2-base',
     ['*.md', '*.txt']),
    ('c) EchoMimicV3 flash-pro (pesos)',
     'BadToBest/EchoMimicV3', f'{BASE}/echomimicv3-flash-pro',
     ['echomimicv3-flash-pro/*']),
]
for nome, repo, destino, padroes in tarefas:
    print(nome, '...')
    try:
        snapshot_download(repo_id=repo, local_dir=destino,
                          allow_patterns=padroes, max_workers=4)
        print('   OK')
    except Exception as e:
        print('   FALHOU:', type(e).__name__, str(e)[:250])
        traceback.print_exc(limit=1)

print(f'\ndownload: {(time.time()-t0)/60:.1f} min')
print('\n=== arquivos grandes baixados ===')
for raiz, _d, arqs in os.walk(BASE):
    for a in arqs:
        p = os.path.join(raiz, a)
        t = os.path.getsize(p)
        if t > 200_000_000:
            print(f'   {t/1e9:6.2f} GB  {p.replace(BASE, ".")}')
print(f'\n/content livre: {shutil.disk_usage("/content").free/1e9:.1f} GB')
print('=== CHECKPOINT 3: OK ===')


In [ ]:
# ── 4. LIGAR OS PESOS AO REPO ──
# O repo espera os modelos em ./flash/<nome>. Em vez de baixar de novo, ligamos por
# symlink (aqui no Colab isso e seguro: tudo vive num disco so, sem coletor de saida).
import glob, shutil

REPO = '/content/em3/echomimic_v3'
os.chdir(REPO)
os.makedirs('flash', exist_ok=True)

mapa = [
    ('Wan2.1-Fun-V1.1-1.3B-InP', 'chinese-wav2vec2-base'),
]
src_base = '/content/em3/pesos'
for item in os.listdir(src_base):
    origem = os.path.join(src_base, item)
    destino = os.path.join(REPO, 'flash', item)
    if os.path.exists(destino):
        print('ja existe:', item)
        continue
    try:
        os.symlink(origem, destino)
        print('ligado:', item)
    except Exception as e:
        print('symlink falhou em', item, e)

print('\n=== conteudo de ./flash ===')
for p in sorted(glob.glob('flash/*')):
    tam = sum(os.path.getsize(os.path.join(r, a))
              for r, _d, fs in os.walk(p) for a in fs) if os.path.isdir(p) else os.path.getsize(p)
    print(f'   {tam/1e9:6.2f} GB  {p}')

print('\n=== scripts de entrada disponiveis ===')
for p in sorted(glob.glob('*.sh')) + sorted(glob.glob('scripts/*.sh')):
    print('   ', p)
print('\n=== o que o run_flash.sh espera (se existir) ===')
try:
    print(open('run_flash.sh').read()[:1200])
except Exception:
    print('   (sem run_flash.sh na raiz)')


In [ ]:
# ── 5. INFERENCIA — o teste que importa ──
resultado = {'sucesso': False, 'tempo_s': None, 'arquivos': [], 'erro': None}

candidatos = ['run_flash.sh', 'scripts/run_flash.sh', 'inference.sh',
              'scripts/inference.sh', 'run_inference.sh', 'scripts/inference_flash.sh']
script = next((c for c in candidatos if os.path.exists(c)), None)
print('script escolhido:', script)

t0 = time.time()
if script:
    rc, saida = sh(f'bash {script} 2>&1 | tail -80', timeout=5400)
else:
    rc, saida = 127, 'sem script conhecido'

resultado['tempo_s'] = round(time.time() - t0, 1)
print('exit code:', rc)

achados = []
for pat in ['outputs/**/*.mp4', '**/*.mp4', 'results/**/*.mp4', '/content/**/*.mp4']:
    achados += glob.glob(pat, recursive=True)
achados = sorted(set(achados))
resultado['arquivos'] = achados
resultado['sucesso'] = rc == 0 and len(achados) > 0
if not achados:
    resultado['erro'] = saida[-700:]

print('\n' + '='*64)
print(f"RESULTADO: sucesso={resultado['sucesso']} | {resultado['tempo_s']}s "
      f"= {resultado['tempo_s']/60:.1f} min | arquivos={len(achados)}")
for a in achados[:6]:
    print('   ', a, os.path.getsize(a), 'bytes')
print('='*64)

json.dump(resultado, open('/content/resultado_echomimic.json', 'w'), indent=1)


In [ ]:
# ── 6. VEREDITO (leia o campo 'sucesso', nao so o tempo) ──
import subprocess, glob, os, json, torch

print('=== GPU usada ===')
print(subprocess.run('nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv,noheader',
                     shell=True, capture_output=True, text=True).stdout.strip())
if torch.cuda.is_available():
    print(f'pico de VRAM: {torch.cuda.max_memory_allocated()/1e9:.2f} GB '
          f'de {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

print('\n=== RESULTADO ===')
try:
    d = json.load(open('/content/resultado_echomimic.json'))
    print(json.dumps(d, indent=1)[:1200])
except Exception as e:
    print('sem arquivo de resultado:', e)

print('\n=== VIDEOS GERADOS ===')
vids = sorted(glob.glob('/content/**/*.mp4', recursive=True))
if vids:
    for v in vids:
        print(f'   {os.path.getsize(v)/1e6:7.2f} MB  {v}')
    print('\n>>> BAIXE O VIDEO ANTES QUE A SESSAO EXPIRE (painel de arquivos a esquerda) <<<')
else:
    print('   nenhum video — leia o erro acima')
